# Lab: MCP Server with DuckDB & Natural Language Queries

## Real-World Scenario

**Challenge:** Your executive director asks questions like:
- "How much did recurring donors give vs. one-time donors?"
- "Who are our most loyal donors?"
- "What's our average gift amount?"

**Without MCP:** You run SQL queries manually, export CSVs, create reports (1-2 hours)

**With MCP + LLM:** Ask natural language questions, get instant answers (30 seconds)

## What We'll Build

1. Load donation CSV into DuckDB
2. Set up MotherDuck MCP server
3. Connect LLM to database via MCP
4. Ask data questions in plain English → get answers

## Setup Instructions

### Step 1: Install DuckDB
```bash
pip install duckdb
```

### Step 2: Install MotherDuck CLI
```bash
pip install motherduck
```

### Step 3: Optional - Get Free MotherDuck Account
- Visit https://motherduck.com
- Create account (free tier available)
- Get authentication token

### Step 4: Keep Ollama Running
```bash
ollama serve
```

## Part 1: Load CSV into DuckDB

In [2]:
import duckdb
import pandas as pd
from pathlib import Path

# Create DuckDB connection (in-memory database)
conn = duckdb.connect(':memory:')

In [5]:
# import duckdb
# import pandas as pd
# from pathlib import Path

# # Create DuckDB connection (in-memory database)
# conn = duckdb.connect(':memory:')

# # Alternatively, use persistent database:
# # conn = duckdb.connect('donations.ddb')

# # Check if week11 data exists
notebook_dir = Path.cwd()
data_path = notebook_dir.parent.parent / 'week11' / 'notebooks' / 'donations.csv'
# if data_path.exists():
#     print(f"✓ Found donation CSV at {data_path}")
# # else:
# #     # Fallback: Create sample data
# #     print("⚠ CSV not found, creating sample data...")
# #     sample_data = """date,donor_id,amount,category,recurring
# # 2024-01-05,101,50.00,One-time,No
# # 2024-01-08,102,25.00,One-time,No
# # 2024-01-12,101,50.00,Recurring,Yes
# # 2024-01-15,103,100.00,One-time,No
# # 2024-02-05,101,50.00,Recurring,Yes
# # 2024-02-10,106,30.00,One-time,No
# # 2024-02-15,102,25.00,Recurring,Yes
# # 2024-03-01,108,45.00,One-time,No
# # 2024-03-05,101,50.00,Recurring,Yes
# # 2024-03-18,109,500.00,One-time,No
# # 2024-04-05,102,25.00,Recurring,Yes
# # 2024-04-15,105,150.00,One-time,No
# # 2024-04-20,101,50.00,Recurring,Yes
# # """
# #     with open('donations_temp.csv', 'w') as f:
# #         f.write(sample_data)
# #     data_path = Path('donations_temp.csv')

# Load CSV into DuckDB table
print("\n📥 Loading CSV into DuckDB...")
conn.execute(f"CREATE TABLE donations AS SELECT * FROM read_csv_auto('{data_path}')")

# Verify the load
result = conn.execute("SELECT COUNT(*) as record_count FROM donations").fetchall()
print(f"✓ Loaded {result[0][0]} donation records")

# Show schema
print("\n📋 Table Schema:")
schema = conn.execute("DESCRIBE donations").fetchall()
for col in schema:
    print(f"  {col[0]}: {col[1]}")

# Show sample data
print("\n📊 Sample Data:")
sample = conn.execute("SELECT * FROM donations LIMIT 5").fetchall()
for row in sample:
    print(f"  {row}")


📥 Loading CSV into DuckDB...
✓ Loaded 100 donation records

📋 Table Schema:
  date: DATE
  donor_id: BIGINT
  amount: DOUBLE
  category: VARCHAR

📊 Sample Data:
  (datetime.date(2024, 1, 1), 138, 217.46672554612113, 'Recurring')
  (datetime.date(2024, 1, 3), 128, 203.49194390602915, 'One-time')
  (datetime.date(2024, 1, 5), 114, 153.80920561183868, 'Recurring')
  (datetime.date(2024, 1, 7), 142, 16.899113130391385, 'One-time')
  (datetime.date(2024, 1, 9), 107, 107.43277800351453, 'Recurring')


## Part 2: Create SQL Query Generator Function

This function will use an LLM to convert natural language questions into SQL queries.

In [6]:
import requests
import json
import re

def query_ollama(prompt, model="llama2", temperature=0.3):
    """
    Send prompt to local Ollama and get response.
    temperature=0.3 for deterministic SQL generation.
    """
    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "temperature": temperature
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()['response'].strip()
    except requests.exceptions.ConnectionError:
        return "[ERROR: Ollama not running. Start with: ollama serve]"
    except Exception as e:
        return f"[ERROR: {str(e)}]"

/Users/anastasiiakulakova/repos/Practical-Data-Science-Course/new_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [7]:
def natural_language_query(question, conn, describe_schema=True):
    """
    Convert natural language question to SQL and execute.
    This demonstrates MCP in action!
    """
    
    # Get table schema if needed
    schema_info = ""
    if describe_schema:
        schema_result = conn.execute("DESCRIBE donations").fetchall()
        schema_info = "\n".join([f"{col[0]}: {col[1]}" for col in schema_result])
    
    # Create prompt for LLM to generate SQL
    prompt = f"""You are a SQL expert for a nonprofit donation database.
Convert this natural language question to a DuckDB SQL query.

TABLE: donations
SCHEMA:
{schema_info}

RULES:
1. Return ONLY valid SQL (no explanation)
2. Use DuckDB syntax
3. Include helpful column aliases
4. Order results sensibly
5. Start response with: SELECT (no markdown, no ```)

QUESTION: {question}

SQL QUERY:"""
    
    print(f"\n🤖 Question: {question}")
    print("\n⏳ LLM generating SQL...")
    
    # Get SQL from LLM
    sql_response = query_ollama(prompt, temperature=0.2)
    
    # Clean up response
    sql = sql_response.strip()
    if sql.startswith("```"):
        sql = sql.split("```")[1].replace("sql", "").strip()
    
    print(f"\n📝 Generated SQL:")
    print(f"   {sql[:100]}..." if len(sql) > 100 else f"   {sql}")
    
    # Execute query
    try:
        print("\n⏳ Executing query...")
        results = conn.execute(sql).fetchall()
        columns = [desc[0] for desc in conn.description]
        
        # Format results
        print(f"\n✅ Results ({len(results)} rows):")
        print(f"   Columns: {', '.join(columns)}")
        for i, row in enumerate(results[:10]):
            print(f"   {row}")
        
        if len(results) > 10:
            print(f"   ... and {len(results) - 10} more rows")
        
        return results, sql
    
    except Exception as e:
        print(f"\n❌ Query execution failed: {e}")
        print(f"   Issue with query: {sql}")
        return None, sql

In [8]:
print("✓ Functions defined")
print("\nReady to answer data questions with LLM + MCP!")

✓ Functions defined

Ready to answer data questions with LLM + MCP!


## Part 3: Ask Natural Language Questions

In [9]:
# Example questions an executive director might ask

questions = [
    "How many total donations have we received?",
    "What is our total amount donated?",
    "How many unique donors do we have?",
    "What's the average donation amount?",
    "How much did recurring donors give vs one-time donors?"
]

print("🎯 EXECUTIVE DIRECTOR QUESTIONS")
print("="*80)
print("\nAsking questions in plain English, getting answers via LLM + DuckDB...\n")

# Ask first question as demo
if len(questions) > 0:
    results, sql = natural_language_query(questions[0], conn)

🎯 EXECUTIVE DIRECTOR QUESTIONS

Asking questions in plain English, getting answers via LLM + DuckDB...


🤖 Question: How many total donations have we received?

⏳ LLM generating SQL...

📝 Generated SQL:
   SELECT COUNT(*) FROM donations;

⏳ Executing query...

✅ Results (1 rows):
   Columns: count_star()
   (100,)


In [10]:
# Ask more questions
for question in questions[1:3]:  # Try a couple more
    results, sql = natural_language_query(question, conn)
    print("\n" + "-"*80)


🤖 Question: What is our total amount donated?

⏳ LLM generating SQL...

📝 Generated SQL:
   SELECT SUM(amount) FROM donations;

⏳ Executing query...

✅ Results (1 rows):
   Columns: sum(amount)
   (24716.21928675364,)

--------------------------------------------------------------------------------

🤖 Question: How many unique donors do we have?

⏳ LLM generating SQL...

📝 Generated SQL:
   SELECT COUNT(DISTINCT donor_id) FROM donations;

⏳ Executing query...

✅ Results (1 rows):
   Columns: count(DISTINCT donor_id)
   (46,)

--------------------------------------------------------------------------------


## Part 4: Understanding MCP in Action

What just happened above is exactly what MCP enables:

In [ ]:
print("MCP ARCHITECTURE IN ACTION")
print("="*80)

architecture = """
┌─────────────────────────────────────────────────────────────────┐
│                    EXECUTIVE DIRECTOR                           │
│         "How much did recurring donors give?"                   │
└────────────┬────────────────────────────────────────────────────┘
             │
             │ Human Question (Natural Language)
             │
             ▼
┌─────────────────────────────────────────────────────────────────┐
│                 LLM (Ollama via MCP)                            │
│  • Receives question                                             │
│  • Understands donation database schema (MCP provides this)      │
│  • Generates appropriate DuckDB SQL                              │
└────────────┬────────────────────────────────────────────────────┘
             │
             │ SQL Query (Machine-readable)
             │
             ▼
┌─────────────────────────────────────────────────────────────────┐
│              MCP Tool: DuckDB Query Executor                    │
│  • Receives SQL from LLM                                         │
│  • Executes against donations table                              │
│  • Returns results                                               │
└────────────┬────────────────────────────────────────────────────┘
             │
             │ Data Results
             │
             ▼
┌─────────────────────────────────────────────────────────────────┐
│                 LLM Formats Response                             │
│  • Takes raw query results                                       │
│  • Formats for human understanding                               │
│  • Adds context and insights                                     │
└────────────┬────────────────────────────────────────────────────┘
             │
             │ Human-Readable Answer
             │
             ▼
┌─────────────────────────────────────────────────────────────────┐
│                    EXECUTIVE DIRECTOR                           │
│  "Recurring donors gave $XXX, one-time donors gave $YYY"        │
│  📊 Gets answer in seconds, not hours!                          │
└─────────────────────────────────────────────────────────────────┘
"""

print(architecture)

print("\nKEY INNOVATIONS:")
print("1. LLM understands database schema via MCP")
print("2. Generates correct SQL without manual intervention")
print("3. DuckDB executes queries fast")
print("4. Results come back in milliseconds")
print("5. Non-technical staff gets instant answers")
print("\n❌ WITHOUT MCP: 'I need to ask IT for a report' (1-2 days)")
print("✅ WITH MCP: 'I get an instant answer' (30 seconds)")

## Part 7: Setting Up MotherDuck MCP Server (Optional)

If you want to use cloud MotherDuck instead of local DuckDB.

In [ ]:
# Instructions for MotherDuck setup

motherduck_setup = """
MOTHERDUCK MCP SERVER SETUP
===========================

Step 1: Create MotherDuck Account
  - Go to https://motherduck.com
  - Sign up (free tier available)
  - Get authentication token

Step 2: Install MotherDuck CLI
  pip install motherduck

Step 3: Authenticate
  motherduck auth
  [Follow prompts to add your token]

Step 4: Start MotherDuck Service
  motherduck serve
  [This runs the MCP server]

Step 5: Connect from Python
  import duckdb
  conn = duckdb.connect('motherduck://')
  
Step 6: Upload Donations Data
  conn.execute("CREATE TABLE donations AS SELECT * FROM read_csv_auto('donation_template.csv')")
  
Step 7: Same queries now run on cloud!
  results = conn.execute("SELECT * FROM donations").fetchall()

BENEFIT:
  • Data persists across sessions
  • Team can share access
  • Query large datasets instantly
  • Same MCP interface as local DuckDB
  • All same natural language capabilities

COST:
  • Free tier: 24GB storage, perfect for NGOs
  • Paid: $20-200/month for enterprise features
"""

print(motherduck_setup)

## Part 8: Exercise - Create Your Own Query

Practice using natural language queries on donation data.

In [ ]:
# YOUR TURN: Ask your own question

# Example questions to try:
my_questions = [
    "What's the largest single donation we've received?",
    "How many donors gave donations of $100 or more?",
    "Show the donation timeline - when did we receive each donation?"
]

print("EXERCISE: Natural Language Data Analysis")
print("="*80)
print("\nTry asking your own questions!\n")

# Pick one to try
my_question = my_questions[0]
print(f"Trying: {my_question}\n")
results, sql = natural_language_query(my_question, conn)

print("\n" + "="*80)
print("\nNow try asking your own questions:")
print("  results, sql = natural_language_query('YOUR QUESTION HERE', conn)")

## Summary: MCP in the Real World

### What We Built
```
CSV Data → DuckDB → LLM (via MCP) → Natural Language Answers
```

### The MCP Magic
- **Without MCP:** Data is locked away. Need SQL knowledge to query.
- **With MCP:** LLM can see the database. Anyone can ask questions.

### Real NGO Workflow
```
Day 1: 
  Executive: "I need donor analysis for the board meeting"
  Staff: "OK, I'll run SQL queries and make a spreadsheet" (2 hours)

With MCP:
  Executive: "I need donor analysis for the board meeting"
  Staff: "Done!" [Uses LLM to query DuckDB] (30 sec)
```

### How This Scales
- **1 question:** 5 min savings
- **20 questions/month:** 100 min = 1.5 hrs saved
- **1 staff person × 12 months:** ~18 hours/year
- **10 staff × 12 months:** **~180 hours/year saved**

### Next Steps
1. Try more queries above
2. Move to MotherDuck when ready for team collaboration
3. Connect to other data sources (grants database, volunteer records, etc)
4. Build dashboard on top of DuckDB

### Key Concept: MCP Makes Data Democratized
✅ Non-technical staff can query databases  
✅ No IT department needed for simple analysis  
✅ LLM becomes smart database assistant  
✅ Faster decisions based on real data  